In [0]:
def rename_column(data):
    data = data.withColumnRenamed("Start date", "start_date")
    data = data.withColumnRenamed("Start station", "start_station")
    data = data.withColumnRenamed("End date", "end_date")
    data = data.withColumnRenamed("End station", "end_station")
    data = data.withColumnRenamed("Bike number", "bike_number")
    data = data.withColumnRenamed("Bike model", "bike_model")
    data = data.withColumnRenamed("Total duration", "total_duration")
    data = data.withColumnRenamed("Total duration (ms)", "total_duration_ms")
    data = data.withColumnRenamed("Start station number", "start_station_number")
    data = data.withColumnRenamed("End station number", "end_station_number")   
    data = data.withColumnRenamed("Number", "number")
    return data



In [0]:
mar01 = spark.read.format("csv").option("header", True).load("/Volumes/workspace/tfl/raw/tfl_cycling_2026/439JourneyDataExtract01Mar2026-15Mar2026.csv")

mar01 = rename_column(mar01)

mar15 = spark.read.format("csv").option("header", True).load("/Volumes/workspace/tfl/raw/tfl_cycling_2026/440JourneyDataExtract15Mar2026-31Mar2026.csv")

mar15 = rename_column(mar15)

jan01 = spark.read.format("csv").option("header", True).load("/Volumes/workspace/tfl/raw/tfl_cycling_2026/435JourneyDataExtract01Jan2026-15Jan2026.csv")

jan01 = rename_column(jan01)

jan15 = spark.read.format("csv").option("header", True).load("/Volumes/workspace/tfl/raw/tfl_cycling_2026/436JourneyDataExtract16Jan2026-31Jan2026.csv")

jan15 = rename_column(jan15)

feb01 = spark.read.format("csv").option("header", True).load("/Volumes/workspace/tfl/raw/tfl_cycling_2026/437JourneyDataExtract01Feb2026-14Feb2026.csv")

feb01 = rename_column(feb01)

feb15 = spark.read.format("csv").option("header", True).load("/Volumes/workspace/tfl/raw/tfl_cycling_2026/438JourneyDataExtract15Feb2026-28Feb2026.csv")

feb15 = rename_column(feb15)

In [0]:
from pyspark.sql.functions import col

def add_source_column(data):
    data = data.withColumn("source_file", col("_metadata.file_path"))
    return data

In [0]:
jan01 = add_source_column(jan01)
jan15 = add_source_column(jan15)
feb01 = add_source_column(feb01)
feb15 = add_source_column(feb15)
mar01 = add_source_column(mar01)
mar15 = add_source_column(mar15)

In [0]:
bronze = jan01.unionByName(jan15, allowMissingColumns=True).unionByName(feb01, allowMissingColumns=True).unionByName(feb15, allowMissingColumns=True).unionByName(mar01, allowMissingColumns=True).unionByName(mar15, allowMissingColumns=True)

In [0]:
from pyspark.sql import functions as F

bronze = bronze.withColumn("ingested_at",F.current_timestamp())

In [0]:
bronze.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.tfl.bronze_journeys")

In [0]:
test = spark.read.table("workspace.tfl.bronze_journeys")


In [0]:
test.count()